## 堆管理：空闲链表与切块策略（ch12 ↔ 动态分区分配）

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- 堆范围 `2048–16383`；块头 = `[next][length]`，`length` 计数据字数，块总长 = `length + 2`
- `alloc`：遍历空闲链表找够大的块 → **总是切块**（Coursera 版算法），剩余部分留在链表上，因此 `freeList` 永不为空，省掉「删除链表中间节点」的双指针逻辑
- 摘除头节点是特例（无前驱），必须单独分支或加哨兵；书版算法（可能整块拿出）与 Coursera 版（总切块）**块头定义不同，不能混用**
- `deAlloc`：从数据区地址回推 `blk = o - 2`，头插法挂回链表（`length` 字段本来就还在）
- 碎片：切块策略天然产生外碎片；改进 = 按地址有序插入 + 相邻块合并（defrag）
- 对应 408：动态分区分配、首次适应 / 最佳适应、碎片与拼接 —— 这就是「内存管理」那一章算法的实物版


## CSAPP malloclab：块布局、四情况合并与 408 的动态分区分配

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.7（实践仓 `learning-408/csapp/malloclab/mm.c`，目前仍是官方 mm-naive 骨架）
块布局（64 位，8 字节对齐）：`header(8: size|alloc) | payload | footer(8: size|a)`；`size` 低 3 位因对齐恒为 0，最低位挪作 alloc 标志；**最小块 32 B**（header+footer+两个指针）；`mem_sbrk` 只增不减。

**四情况合并**（`mm_free` 时立即做）：前空/后空组合成 4 个分支，靠前块 footer 与后块 header 的 alloc 位判断；合并后必须保证**堆里不存在两个相邻空闲块**。

**408 对应（几乎同一张图）**：first-fit / next-fit / best-fit ↔ **首次适应 / 邻近适应 / 最佳适应 / 最坏适应**（必考）；合并 ↔ 紧凑与碎片整理；对齐浪费 ↔ 内部碎片；利用率 U < 1 ↔ 外部碎片。


## 分配器的三级演进与评分公式：不许走极端

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.7.4–7.7.8
三级演进：① **隐式空闲链表**（扫所有块）≈ Perf 50–52；② **显式空闲链表**（空闲块里放 pred/succ，只扫空闲块，LIFO 插表头）≈ mid-80s；③ **分离空闲链表**（按大小类分 10–14 个桶，近似 best-fit 且定位 O(1)）→ 冲高分。

评分（官方 3e）：正确性 20 + 性能 35 + 风格 10，
`P = w*U + (1-w)*min(1, T/T_libc)`，`w = 0.6`
U:`P = 0.6·U + 0.4·min(1, T/T_libc)`

两个指标各最多贡献 w 与 1-w，**官方明说不许走极端** —— 这条公式本身就是「先定可测量指标，再谈优化」的范例。

**工程纪律**：指针算术全封进宏（`HDRP/FTRP/NEXT_BLKP/PREV_BLKP/GET_SIZE/GET_ALLOC`）；**先写 `mm_checkheap`（不变量清单）再写实现**；先过前 9 个 trace（只有 malloc/free），再动后 2 个（含 realloc）。
